# Transformer Encoder — From Scratch with PyTorch

## 1. Input

For the toy implementation:

vocab_size = 20
embedding_dim = 16
batch_size = 4
sequence_length = 8

The input should contain TOKEN IDs, not already-embedded vectors.

Example:

x.shape = [batch_size, sequence_length]
       = [4, 8]

Each value is an integer representing a token in the vocabulary.

Example:
[3, 7, 2, 15, 4, 9, 1, 6]


## 2. Word Embedding

nn.Embedding(vocab_size, embedding_dim)

Converts each token ID into a dense vector.

vocab_size = 20
embedding_dim = 16

Therefore:

[4, 8]
   ↓
[4, 8, 16]

Each of the 8 tokens is represented by a 16-dimensional vector.

The embedding layer learns these vectors during training.


## 3. Positional Encoding

Transformers do not inherently know the order of tokens.

Positional encoding adds information about where each token occurs in the sequence.

For every position, a 16-dimensional positional vector is created.

The positional encoding has shape:

[1, max_len, d_model]

For our input:

x.shape = [4, 8, 16]

Positional encoding:

[1, 8, 16]

Because of broadcasting:

[4, 8, 16] + [1, 8, 16]

becomes:

[4, 8, 16]

The embedding and positional encoding therefore remain the same size.


## 4. Self-Attention

Each token creates three representations:

Q = Query
K = Key
V = Value

These are produced using learned linear transformations:

Q = W_q(x)
K = W_k(x)
V = W_v(x)

For:

x.shape = [4, 8, 16]

we get:

Q.shape = [4, 8, 16]
K.shape = [4, 8, 16]
V.shape = [4, 8, 16]


### Attention scores

K is transposed in its final two dimensions:

K.transpose(-2, -1)

[4, 8, 16]
    ↓
[4, 16, 8]

Then:

Q @ Kᵀ

[4, 8, 16] @ [4, 16, 8]

= [4, 8, 8]

This gives an attention score between every pair of tokens.

For each sequence:

8 query tokens × 8 key tokens

Therefore the attention matrix is:

[8, 8]


### Scaling

The attention scores are divided by:

sqrt(d_k)

This prevents the dot products from becoming too large and making softmax unstable.

### Softmax

Softmax is applied across the key/token dimension:

softmax(attention_scores, dim=-1)

This converts the scores into attention weights.

For each query token, the weights across all tokens sum to 1.


### Weighted values

The attention weights are multiplied by V:

[4, 8, 8] @ [4, 8, 16]

= [4, 8, 16]

The output has the same shape as the original token representations.


## 5. Important: Current Implementation Is Single-Head Attention

Although the class was called MultiHeadAttention, the implementation initially performs attention over the entire 16-dimensional representation at once.

It does NOT actually split the representation into multiple heads.

For real multi-head attention:

d_model = 16
num_heads = 4

Then:

head_dim = 16 / 4 = 4

The 16-dimensional representation is divided into 4 separate attention heads.

Each head works with 4 dimensions.

The heads are eventually concatenated back together:

4 heads × 4 dimensions = 16 dimensions


## 6. Feed Forward Network

After attention, every token representation independently passes through a feed-forward neural network.

For example:

16 → 4 → 16

First linear layer:

[4, 8, 16]
     ↓
[4, 8, 4]

ReLU activation

Then second linear layer:

[4, 8, 4]
     ↓
[4, 8, 16]

The sequence length and embedding dimension are restored.


## 7. Residual Connections

The transformer adds the original representation back to the transformed representation.

After attention:

x = x + attention_output

This is called a residual/skip connection.

Purpose:
- helps information flow through deeper networks
- makes optimization easier
- prevents the transformation from having to completely recreate the original information


## 8. Layer Normalization

After the residual connection:

x = LayerNorm(x)

This normalizes the representation of each token.

The shape does not change:

[4, 8, 16] → [4, 8, 16]


## 9. Transformer Encoder Block

The complete encoder block is:

Input
  ↓
Embedding
  ↓
Positional Encoding
  ↓
Self-Attention
  ↓
Residual Connection
  ↓
LayerNorm
  ↓
Feed Forward Network
  ↓
Residual Connection
  ↓
LayerNorm
  ↓
Output


## 10. Classification

The transformer produces a representation for every token:

x.shape = [batch_size, sequence_length, d_model]

For classification, one representation can be selected.

In the implementation:

last_output = x[:, -1, :]

For:

x.shape = [4, 8, 16]

this gives:

last_output.shape = [4, 16]

This takes the representation of the final token for each sequence.


## 11. Classifier

The final token representation is passed through a linear layer:

nn.Linear(16, 2)

Therefore:

[4, 16]
   ↓
[4, 2]

The two values are the logits for the two classes.


## 12. Loss Function

For two-class classification:

criterion = nn.CrossEntropyLoss()

The model outputs logits:

[batch_size, number_of_classes]

For this implementation:

[4, 2]

The target contains the class index for each example:

y.shape = [4]

Example:

y = [0, 1, 1, 0]

CrossEntropyLoss compares the predicted logits with the correct class.


## 13. Training Loop

Each training iteration follows:

1. Forward pass

logits = model(x)

2. Calculate loss

loss = criterion(logits, y)

3. Clear old gradients

optimizer.zero_grad()

4. Backpropagation

loss.backward()

5. Update parameters

optimizer.step()


## 14. Why zero_grad() is needed

Gradients in PyTorch accumulate by default.

Therefore, before calculating gradients for the next iteration:

optimizer.zero_grad()

clears the gradients from the previous iteration.

Then:

loss.backward()

calculates the new gradients.

Finally:

optimizer.step()

uses those gradients to update the model parameters.


## 15. Important Tensor Shape Summary

Initial token IDs:

[batch, sequence]
[4, 8]

After embedding:

[batch, sequence, d_model]
[4, 8, 16]

After positional encoding:

[4, 8, 16]

Q, K, V:

[4, 8, 16]

Attention scores:

[4, 8, 8]

Attention output:

[4, 8, 16]

After feed-forward network:

[4, 8, 16]

Last token:

[4, 16]

Classifier output:

[4, 2]

Target:

[4]


## 16. Important Concept

The transformer does NOT reduce the entire sequence to one vector during the encoder block.

It maintains:

one d_model-dimensional representation for EVERY token.

For our example:

8 tokens × 16 features

= 8 token representations, each with 16 features.

Only at the classification stage do we select one representation (the final token in this implementation) to produce the class prediction.


## 17. What This Implementation Demonstrates

This notebook implements the main components of a transformer encoder:

- Token embedding
- Sinusoidal positional encoding
- Query, Key, Value projections
- Scaled dot-product self-attention
- Residual connections
- Layer normalization
- Feed-forward network
- Dropout
- Final classification layer
- Cross-entropy loss
- Backpropagation
- Parameter updates

The attention implementation is currently single-head and can be extended to true multi-head attention.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import math

In [36]:
vocab_size = 20
embedding_dim = 16
batch_size = 4
sequence_length = 8
x = torch.randint(0, vocab_size, (batch_size, sequence_length))
y = torch.randint(0, 2, (4,))

In [37]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model=16, max_len=50):
        super(PositionalEncoding, self).__init__()

        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2)
            * (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(max_len, d_model)

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

In [42]:
class SelfAttention(nn.Module):
    def __init__(self, d_model = 16):
        super(SelfAttention, self).__init__()

        self.d_k = d_model


        # Linear layers for Q, K, V transformations
        self.W_q = nn.Linear(d_model, d_model, bias = False)
        self.W_k = nn.Linear(d_model, d_model, bias = False)
        self.W_v = nn.Linear(d_model, d_model, bias = False)
        self.fc_out = nn.Linear(d_model, d_model)

    def forward(self, query, key, value):
        batch_size = query.size(0)

        # Perform linear transformations and split into heads
        Q = self.W_q(query)
        K = self.W_k(key)
        V = self.W_v(value)

        # Compute attention
        attn_weights = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn_weights = torch.softmax(attn_weights, dim=-1)
        attn_output = torch.matmul(attn_weights, V)

        # Concatenate heads and put through final linear layer
        return self.fc_out(attn_output)


In [39]:
class FeedForwardNetwork(nn.Module):
    def __init__(self, d_model = 16, d_ff = 4):
        super(FeedForwardNetwork, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(0.1)
        self.activation = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.dropout(self.activation(self.fc1(x))))

In [46]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_ff = 4, dropout=0.1, d_model = 16, max_len = 50):
        super(TransformerEncoderLayer, self).__init__()
        self.we = nn.Embedding(vocab_size, embedding_dim)
        self.pe = PositionalEncoding(d_model = d_model, max_len = max_len)
        self.self_attn = SelfAttention(d_model)
        self.ffn = FeedForwardNetwork(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, 2)

    def forward(self, x):

        x = self.we(x)
        x = self.pe(x)
        # Self-Attention with Residual Connection
        x = x + self.dropout(self.self_attn(x, x, x))
        x = self.norm1(x)

        # Feedforward with Residual Connection
        x = x + self.dropout(self.ffn(x))
        x = self.norm2(x)
        last_output = x[:, -1, :]
        self.logits = self.classifier(last_output)

        return x


In [47]:
model = TransformerEncoderLayer()
optimizer = optim.SGD(model.parameters(), lr=0.1)
criterion = nn.CrossEntropyLoss()
epochs = 10
for epoch in range(epochs):
    output = model(x)
    loss = criterion(model.logits, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [1/10], Loss: 0.3681
Epoch [2/10], Loss: 0.2568
Epoch [3/10], Loss: 0.2280
Epoch [4/10], Loss: 0.1745
Epoch [5/10], Loss: 0.1488
Epoch [6/10], Loss: 0.1152
Epoch [7/10], Loss: 0.1023
Epoch [8/10], Loss: 0.1190
Epoch [9/10], Loss: 0.0832
Epoch [10/10], Loss: 0.0870
